# 📓 실습 02. 시계열 센서 데이터 드리프트 통계적 검정 및 PSI 계산

이 실습에서는 CMP 공정의 정상 센서 데이터셋(Reference)과 장비 노후화로 인해 노이즈 및 편차가 가미된 시점의 센서 데이터셋(Current)을 구성한 뒤, 통계량 검정인 **KS-Test**와 **PSI(Population Stability Index)** 지표를 수학적으로 작동시켜 모델 성능 저하 징후를 탐지합니다.

In [ ]:
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt

# 1. 정상 데이터(Reference) 및 드리프트 데이터(Current) 무작위 설계
np.random.seed(42)
ref_flow = np.random.normal(loc=120, scale=5, size=500) # 평균 120
curr_flow = np.random.normal(loc=112, scale=7, size=500) # 센서 노후화로 평균이 112로 감소하고 튐

print(f"참조 데이터 평균: {np.mean(ref_flow):.2f} (표준편차: {np.std(ref_flow):.2f})")
print(f"현재 데이터 평균: {np.mean(curr_flow):.2f} (표준편차: {np.std(curr_flow):.2f})")

In [ ]:
# 2. Kolmogorov-Smirnov Test (KS-Test) 검정 수행
ks_stat, p_val = stats.ks_2samp(ref_flow, curr_flow)
print("=== [KS-Test 검정 결과] ===")
print(f"KS 통계량 (D-value): {ks_stat:.4f}")
print(f"유의 확률 (p-value): {p_val:.6e}")

if p_val < 0.05:
    print("🚨 [결과] 두 데이터 분포가 유의미하게 달라졌습니다! 데이터 드리프트 경보 발령!")
else:
    print("✅ [결과] 데이터 분포가 안정적입니다.")

In [ ]:
# 3. PSI (Population Stability Index) 계산 함수 구현
def calculate_psi(expected, actual, num_buckets=10):
    # 구간 나누기
    percentiles = np.linspace(0, 100, num_buckets + 1)
    buckets = np.percentile(expected, percentiles)
    buckets[0] = -np.inf
    buckets[-1] = np.inf
    
    psi_value = 0.0
    for i in range(num_buckets):
        # 버킷별 데이터 개수 척도
        e_cnt = np.sum((expected > buckets[i]) & (expected <= buckets[i+1]))
        a_cnt = np.sum((actual > buckets[i]) & (actual <= buckets[i+1]))
        
        # 비율 변환
        e_pct = max(0.0001, e_cnt / len(expected))
        a_pct = max(0.0001, a_cnt / len(actual))
        
        # PSI 가산
        psi_value += (a_pct - e_pct) * np.log(a_pct / e_pct)
        
    return psi_value

psi = calculate_psi(ref_flow, curr_flow)
print(f"\nPSI 계산 결과: {psi:.4f}")
if psi >= 0.25:
    print("🚨 [경보] PSI가 0.25를 넘었습니다. 즉각적인 재학습 파이프라인 트리거 필요!")
elif psi >= 0.1:
    print("⚠️ [주의] 미세한 분포 변화 감지.")
else:
    print("✅ [안정] 정상 수치 범위 내 상태 유지.")